<a href="https://colab.research.google.com/github/workshahnawaz643-tech/Enayat-GLaucoma/blob/main/CLAHRetinexVessel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opencv-python scikit-image matplotlib numpy

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
from skimage.filters import frangi
from skimage import exposure

# To run this program. Download the dataset from kaggle "https://www.kaggle.com/datasets/arnavjain1/glaucoma-datasets". Then upload to your google drive and provide the path below.
#The size of dataset (fundus images) are too large to upload in colab so follow this procedure.
drive.mount('/content/drive')

base_dir = "/content/drive/MyDrive/G1020"
input_dir = os.path.join(base_dir, "Images")
output_dir = os.path.join(base_dir, "preprocessed_outputs")
os.makedirs(output_dir, exist_ok=True)

print("Input:", input_dir)
print("Output:", output_dir)


# PREPROCESSING METHODS

# Green channel
def green_channel(img):
    return img[:,:,1]

# CLAHE enhancement
def apply_clahe(gray):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(gray)

# Retinex (illumination correction)
def single_scale_retinex(img, sigma=30):
    img = np.float32(img) + 1.0
    blur = cv2.GaussianBlur(img, (0,0), sigma)
    retinex = np.log(img) - np.log(blur + 1e-5)
    return cv2.normalize(retinex, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

# Vessel enhancement (Frangi filter)
def vessel_enhance(gray):
    vessel = frangi(gray/255.)
    vessel = cv2.normalize(vessel, None, 0, 255, cv2.NORM_MINMAX)
    return vessel.astype(np.uint8)

# Adaptive fusion
def adaptive_fusion(green, clahe, retinex, vessel):

    g = green / 255.
    c = clahe / 255.
    r = retinex / 255.
    v = vessel / 255.


    wg = np.var(g)
    wc = np.var(c)
    wr = np.var(r)
    wv = np.var(v)

    total = wg + wc + wr + wv + 1e-6

    fused = (wg*g + wc*c + wr*r + wv*v) / total
    fused = np.clip(fused*255, 0, 255).astype(np.uint8)

    return fused

# IMAGES

count = 0

for img_name in os.listdir(input_dir):
    if not img_name.lower().endswith((".jpg",".png",".jpeg")):
        continue

    path = os.path.join(input_dir, img_name)
    img = cv2.imread(path)

    if img is None:
        continue


    img = cv2.resize(img, (512,512))

    #  Green channel
    green = green_channel(img)

    #  CLAHE
    clahe = apply_clahe(green)

    #  Retinex
    retinex = single_scale_retinex(green)

    # Vessel enhancement
    vessel = vessel_enhance(green)

    #  Adaptive Fusion
    fused = adaptive_fusion(green, clahe, retinex, vessel)

    # results
    cv2.imwrite(os.path.join(output_dir, f"{img_name}_green.png"), green)
    cv2.imwrite(os.path.join(output_dir, f"{img_name}_clahe.png"), clahe)
    cv2.imwrite(os.path.join(output_dir, f"{img_name}_retinex.png"), retinex)
    cv2.imwrite(os.path.join(output_dir, f"{img_name}_vessel.png"), vessel)
    cv2.imwrite(os.path.join(output_dir, f"{img_name}_fused.png"), fused)


    if count < 3:
        plt.figure(figsize=(12,6))

        plt.subplot(2,3,1); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title("Original"); plt.axis("off")
        plt.subplot(2,3,2); plt.imshow(green, cmap='gray'); plt.title("Green"); plt.axis("off")
        plt.subplot(2,3,3); plt.imshow(clahe, cmap='gray'); plt.title("CLAHE"); plt.axis("off")
        plt.subplot(2,3,4); plt.imshow(retinex, cmap='gray'); plt.title("Retinex"); plt.axis("off")
        plt.subplot(2,3,5); plt.imshow(vessel, cmap='gray'); plt.title("Vessel"); plt.axis("off")
        plt.subplot(2,3,6); plt.imshow(fused, cmap='gray'); plt.title("Fused (Proposed)"); plt.axis("off")

        plt.tight_layout()
        plt.show()

    count += 1